# Volatility Models for Market Risk — walkthrough

This notebook is the interview demo. It runs the whole project top to bottom on **real** data
(Nifty 50, S&P 500, Reliance Industries, daily, 2015 → latest) and explains each step.

* **Part 1** – stylized facts: why constant volatility is wrong.
* **Part 2** – models: rolling windows, EWMA, GARCH(1,1) (by hand and with `arch`), GARCH‑t, GJR‑GARCH, Parkinson.
* **Part 3** – out-of-sample forecast evaluation: MSE, QLIKE, Mincer–Zarnowitz, ranking, COVID zoom.

Every computation lives in `src/`; the notebook calls the same functions as `main.py`, so the numbers here are the numbers in the docs.

In [ ]:
# Make the project root importable (the notebook lives in notebooks/)
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import Image, display

import main as pipeline
from src.config import Config

pd.set_option("display.precision", 4)
cfg = Config()
cfg.output_dir.mkdir(parents=True, exist_ok=True)
results = {}
OUT = cfg.output_dir

## Part 0 — Data

* Download daily OHLC + adjusted close with `yfinance` **once**, cache as CSV in `data/`, reuse afterwards.
* Clean: drop rows with missing / non-positive prices or `High == Low` (stale quote).
* Log returns $r_t = \ln(P_t/P_{t-1})$ on the adjusted close.
* Annualise with $\sigma_{annual} = \sigma_{daily}\sqrt{252}$.

If neither the download nor the cached CSVs are available, the next cell raises an error: synthetic data is never used for results.

In [ ]:
data = pipeline.part0_data(cfg, results)
pd.read_csv(OUT / "data_cleaning.csv", index_col=0)

## Part 1 — Stylized facts

A constant-volatility normal model implies: skewness 0, excess kurtosis 0, no autocorrelation in squared returns, and no link between today's sign and tomorrow's size. We test all four.

In [ ]:
pipeline.part1_stylized_facts(data, cfg, results)
facts = pd.read_csv(OUT / "stylized_facts.csv", index_col=0)
facts[["n_obs", "vol_annual", "skewness", "excess_kurtosis", "jb_stat", "jb_pvalue",
       "obs_4sigma_days", "exp_4sigma_days_normal"]]

**Fat tails.** Excess kurtosis far above 0 and Jarque–Bera p-values ≈ 0 reject normality. The 4‑sigma count compares observed extreme days with what a normal distribution predicts.

In [ ]:
display(Image(OUT / "02_hist_vs_normal.png"))
display(Image(OUT / "03_qq_plot.png"))

**Volatility clustering.** Big moves come in bursts: |r| shows calm and turbulent regimes. Returns themselves are nearly uncorrelated, but squared returns are strongly autocorrelated — Ljung–Box rejects "no autocorrelation" for $r_t^2$ much more strongly than for $r_t$.

In [ ]:
display(Image(OUT / "01_returns_clustering.png"))
display(Image(OUT / "04_acf_returns_vs_squared.png"))
pd.read_csv(OUT / "ljung_box.csv")

**Leverage effect.** A negative $\text{corr}(r_t, r_{t+1}^2)$ means falls are followed by larger moves than rises.

In [ ]:
facts[["corr_r_next_r2", "corr_r_next_absr"]]

**Conclusion.** Volatility is not constant: it clusters and is predictable from recent squared returns, tails are fat, and down-moves raise future volatility more than up-moves. A model with a single σ for the whole sample under-states risk in turbulent periods and over-states it in calm ones. We need models where σ²ₜ depends on recent returns: EWMA and the GARCH family (with fat-tailed and asymmetric variants).

## Part 2.1 — Historical (rolling) volatility

$\sigma_t(N) = \sqrt{\tfrac{1}{N-1}\sum_{i=0}^{N-1}(r_{t-i}-\bar r)^2}$ — equal weight on the last N days.
Short windows are responsive but noisy; long windows are smooth but slow. The **ghost effect**: a single extreme day keeps the estimate high for exactly N days, then it drops abruptly when the day leaves the window.

In [ ]:
pipeline.part2_historical(data, cfg, results)
pd.DataFrame({a: v["avg_abs_daily_change"] for a, v in results["historical"].items()}).T

In [ ]:
display(Image(OUT / "05_hist_vol_SP500.png"))

## Part 2.2 — EWMA (RiskMetrics)

$\sigma^2_t = \lambda\sigma^2_{t-1} + (1-\lambda) r^2_{t-1}$, λ = 0.94. Initialisation: $\sigma^2_1 = r_0^2$ (no future data; its weight decays like $\lambda^t$).
Below: the explicit loop and pandas `.ewm(adjust=False)` give the same numbers.

In [ ]:
from src.models.ewma import ewma_variance_loop, ewma_variance_pandas
r_sp = data["SP500"]["ret"]
loop = ewma_variance_loop(r_sp, 0.94)
fast = ewma_variance_pandas(r_sp, 0.94)
print("max |loop - pandas| =", float(np.nanmax(np.abs(loop - fast))))
assert np.allclose(loop, fast, equal_nan=True)

In [ ]:
pipeline.part2_ewma(data, cfg, results)
display(Image(OUT / "06_ewma_weights.png"))
display(Image(OUT / "07_ewma_lambdas_SP500.png"))
results["ewma"]["half_life_days"]

## Part 2.3 — GARCH(1,1) by hand

$\sigma^2_t = \omega + \alpha r^2_{t-1} + \beta\sigma^2_{t-1}$, with normal log-likelihood
$\ln L = -\tfrac12\sum_t [\ln 2\pi + \ln\sigma_t^2 + r_t^2/\sigma_t^2]$ maximised by `scipy.optimize.minimize` (SLSQP) subject to ω > 0, α, β ≥ 0, α + β < 1.
Returns are multiplied by 100 so that ω is ~0.01–0.1 instead of ~1e‑6 (better-conditioned optimisation). Using the same backcast (sample variance) as the manual version, `arch` gives the same estimates.

In [ ]:
pipeline.part2_garch_manual(data, cfg, results)
pd.read_csv(OUT / "garch_manual_vs_arch.csv", index_col=0)

## Part 2.4 — GARCH with `arch`: normal vs Student‑t, GJR, term structure

Key numbers: persistence α+β, long-run variance $V_L=\omega/(1-\alpha-\beta)$, long-run annual vol, half-life $\ln 0.5/\ln(\alpha+\beta)$.
EWMA is the special case ω = 0, α = 1−λ, β = λ (α+β = 1: no mean reversion, no long-run level) — asserted in `part2_ewma`.

In [ ]:
pipeline.part2_garch_arch(data, cfg, results)
summary = pd.read_csv(OUT / "garch_summary.csv", index_col=0)
summary[["normal_persistence", "normal_long_run_annual_vol", "normal_half_life_days",
         "t_persistence", "t_nu", "next_day_vol_annual"]]

In [ ]:
params = pd.read_csv(OUT / "garch_params.csv", index_col=0)
params[params["asset"] == "SP500"]

**Normal vs Student-t errors.** Higher log-likelihood and lower AIC/BIC favour the t; a small ν (degrees of freedom) means fat tails remain even after GARCH removes the clustering.

In [ ]:
pd.read_csv(OUT / "garch_information_criteria.csv", index_col=0)

**Term structure.** $E[\sigma^2_{t+h}] = V_L + (\alpha+\beta)^{h-1}(\sigma^2_{t+1}-V_L)$: forecasts mean-revert to the long-run level (manual formula asserted equal to `arch`).

In [ ]:
display(Image(OUT / "08_garch_term_structure.png"))

**GJR-GARCH and the leverage effect.** $\sigma^2_t = \omega + (\alpha + \gamma I_{t-1}) r^2_{t-1} + \beta\sigma^2_{t-1}$, $I_{t-1}=1$ if $r_{t-1}<0$. γ > 0 and significant (t-stat, p-value) ⇒ bad news raises variance more than good news. The news impact curve becomes steeper on the left.

In [ ]:
summary[["gjr_alpha", "gjr_gamma", "gjr_gamma_t", "gjr_gamma_p", "gjr_beta", "gjr_persistence"]]

In [ ]:
display(Image(OUT / "09_news_impact_curve.png"))

## Part 2.5 — Parkinson range-based volatility

$\sigma^2_P = \frac{1}{4\ln 2}(\ln H/L)^2$ uses the intraday range (≈5× more efficient than close-to-close for a driftless random walk) but misses overnight gaps, so it is biased down where gaps are large.

In [ ]:
pipeline.part2_parkinson(data, cfg, results)
pd.DataFrame(results["parkinson"]).T

In [ ]:
display(Image(OUT / "10_parkinson_SP500.png"))

## Part 3 — Out-of-sample forecast evaluation

* Expanding window: first 1000 returns for estimation only; GARCH-type models refit every 20 days on data **before** the forecast date — no look-ahead (see `tests/test_no_lookahead.py`).
* Proxy: $RV_t = r_t^2$. Losses: MSE $=\overline{(RV-h)^2}$, QLIKE $=\overline{RV/h + \ln h}$ (robust ranking with a noisy proxy, not dominated by a few crash days).
* Mincer–Zarnowitz: $RV_t = a + b h_t + e_t$; unbiased forecasts have a = 0, b = 1.

This cell takes a few minutes (≈900 GARCH fits).

In [ ]:
pipeline.part3_evaluation(data, cfg, results)

In [ ]:
ranking = pd.read_csv(OUT / "model_ranking.csv", index_col=0)
ranking

In [ ]:
pd.read_csv(OUT / "rank_summary_qlike.csv", index_col=0)

In [ ]:
display(Image(OUT / "13_qlike_by_model.png"))
display(Image(OUT / "14_mse_by_model.png"))

In [ ]:
mz = pd.read_csv(OUT / "mincer_zarnowitz.csv", index_col=0)
mz

### COVID-19 crash zoom

How fast does each model react to the February–March 2020 sell-off, and how fast does it decay afterwards?

In [ ]:
display(Image(OUT / "12_covid_zoom_SP500.png"))
display(Image(OUT / "12_covid_zoom_NIFTY50.png"))
pd.read_csv(OUT / "covid_reaction.csv", index_col=0)

In [ ]:
display(Image(OUT / "11_oos_forecasts_SP500.png"))
vf = pd.read_csv(OUT / "vol_forecasts.csv")
print(vf.shape)
vf.head()

## Save the key numbers

`outputs/key_results.json` holds every number quoted in README / INTERVIEW_DEFENSE.

In [ ]:
import json
with open(OUT / "key_results.json", "w") as handle:
    json.dump(results, handle, indent=1, default=str)
print("saved", OUT / "key_results.json")